# NBA Fantasy Points Predictor

Predicting DraftKings fantasy points per player per game.

**DK scoring:** PTS + 1.25*REB + 1.5*AST + 2*STL + 2*BLK - 0.5*TOV + 0.5*FG3M

Features come from `data/nba_features.csv` (built by `build_features.py`).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

DATA_DIR = Path("data")
MODEL_DIR = Path("models")

## Load processed features

`nba_features.csv` is one row per player-game. The manifest tells us which columns are identifiers, the target, component stats, or model features.

In [2]:
df = pd.read_csv(DATA_DIR / "nba_features.csv", parse_dates=["GAME_DATE"])
manifest = json.load(open(DATA_DIR / "nba_features_manifest.json"))

feature_cols = []
for cols in manifest["groups"].values():
    feature_cols += cols
feature_cols = list(dict.fromkeys(feature_cols))

target = manifest["target"]
components = manifest["target_components"]

print(f"rows:       {len(df):,}")
print(f"features:   {len(feature_cols)}")
print(f"target:     {target}")
print(f"components: {components}")

rows:       563,774
features:   107
target:     FANTASY_PTS
components: ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FG3M']


## Train / test split

Split by date so we don't train on future games. Everything before the 2023-24 season is train, the rest is test.

In [3]:
is_train = df["GAME_DATE"] < "2023-10-01"
is_test  = ~is_train

X_train = df.loc[is_train,  feature_cols]
X_test  = df.loc[is_test,   feature_cols]
y_train = df.loc[is_train,  target]
y_test  = df.loc[is_test,   target]

print(f"train: {len(X_train):,} rows")
print(f"test:  {len(X_test):,} rows")

train: 497,892 rows
test:  65,882 rows


## Multi-output decomposition

Instead of predicting `FANTASY_PTS` directly, we train one HistGradientBoosting model per component stat (PTS, REB, AST, STL, BLK, TOV, FG3M) and combine the predictions with the DK formula. Each model can specialize on its stat -- rebounds depend on size, assists on role, blocks on position -- and the DK weights amplify the most important components.

In [4]:
DK_WEIGHTS = {
    "PTS":  1.00,
    "REB":  1.25,
    "AST":  1.50,
    "STL":  2.00,
    "BLK":  2.00,
    "TOV": -0.50,
    "FG3M": 0.50,
}

## Train with sklearn defaults

First pass uses `HistGradientBoostingRegressor` with the parameters we settled on by hand.

In [5]:
y_train_components = {s: df.loc[is_train, s].astype(float).values for s in components}
y_test_components  = {s: df.loc[is_test,  s].astype(float).values for s in components}

default_params = dict(
    max_iter=500,
    learning_rate=0.05,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
)

pred_tr = np.zeros(len(X_train))
pred_te = np.zeros(len(X_test))

for stat, weight in DK_WEIGHTS.items():
    m = HistGradientBoostingRegressor(**default_params)
    m.fit(X_train, y_train_components[stat])
    ptr = m.predict(X_train)
    pte = m.predict(X_test)
    pred_tr += weight * ptr
    pred_te += weight * pte
    rmse_te = np.sqrt(mean_squared_error(y_test_components[stat], pte))
    print(f"  {stat:5s} (w={weight:+.2f})  test RMSE={rmse_te:.3f}")

default_train_rmse = np.sqrt(mean_squared_error(y_train, pred_tr))
default_test_rmse  = np.sqrt(mean_squared_error(y_test,  pred_te))
print()
print(f"FANTASY_PTS train RMSE: {default_train_rmse:.3f}")
print(f"FANTASY_PTS test  RMSE: {default_test_rmse:.3f}")

  PTS   (w=+1.00)  test RMSE=6.100


  REB   (w=+1.25)  test RMSE=2.539


  AST   (w=+1.50)  test RMSE=1.914


  STL   (w=+2.00)  test RMSE=0.969


  BLK   (w=+2.00)  test RMSE=0.783


  TOV   (w=-0.50)  test RMSE=1.214


  FG3M  (w=+0.50)  test RMSE=1.307

FANTASY_PTS train RMSE: 9.064
FANTASY_PTS test  RMSE: 9.551


## Optuna best params (if available)

If a finished Optuna study is on disk under `models/run_*/optuna_study_results.json`, load the best params, retrain the seven HGB models, and report the new RMSE. Otherwise just print a note and skip.

In [6]:
study_files = sorted(MODEL_DIR.glob("run_*/optuna_study_results.json"))

if study_files:
    latest = study_files[-1]
    print(f"using {latest}")
    with open(latest) as f:
        study = json.load(f)

    bp = study["best_params"]
    tuned_params = dict(
        loss="squared_error",
        learning_rate=bp.get("learning_rate", 0.05),
        max_iter=bp.get("max_iter", 500),
        max_depth=bp.get("max_depth", 8),
        min_samples_leaf=bp.get("min_samples_leaf", 20),
        l2_regularization=bp.get("l2_regularization", 0.0),
        max_bins=bp.get("max_bins", 255),
        max_leaf_nodes=bp.get("max_leaf_nodes") if bp.get("use_max_leaf_nodes", False) else None,
        early_stopping=True,
        n_iter_no_change=bp.get("n_iter_no_change", 15),
        validation_fraction=bp.get("validation_fraction", 0.15),
        random_state=42,
    )

    pred_tr_t = np.zeros(len(X_train))
    pred_te_t = np.zeros(len(X_test))
    for stat, weight in DK_WEIGHTS.items():
        m = HistGradientBoostingRegressor(**tuned_params)
        m.fit(X_train, y_train_components[stat])
        pred_tr_t += weight * m.predict(X_train)
        pred_te_t += weight * m.predict(X_test)

    tuned_train_rmse = np.sqrt(mean_squared_error(y_train, pred_tr_t))
    tuned_test_rmse  = np.sqrt(mean_squared_error(y_test,  pred_te_t))
    print(f"Optuna train RMSE: {tuned_train_rmse:.3f}")
    print(f"Optuna test  RMSE: {tuned_test_rmse:.3f}")
else:
    tuned_test_rmse = None
    print("No Optuna study found - reporting default-params result only.")

No Optuna study found - reporting default-params result only.


## Final results

In [7]:
rows = [
    {"model": "HistGB multi-output (sklearn defaults)", "test_rmse": default_test_rmse},
]
if tuned_test_rmse is not None:
    rows.append({"model": "HistGB multi-output (Optuna best)", "test_rmse": tuned_test_rmse})

summary = pd.DataFrame(rows).sort_values("test_rmse").reset_index(drop=True)
print(summary.to_string(index=False))

                                 model  test_rmse
HistGB multi-output (sklearn defaults)   9.551426
